# 04 – Train/test split

The cleaned data is split once into a training set (70%) and a test set (30%). The test set is not used again until the final evaluation. The split is stratified by failure mode so that the failure rate and the mix of failure modes are the same in both sets. Jobs with several failure modes are assigned to the first mode in the dataset's column order (TWF, HDF, PWF, OSF, RNF), because some combinations occur only once and a group must contain at least two jobs to be split.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from config import CLEAN_DATA, TRAIN_IDS, TEST_IDS, SEED, TEST_SIZE, TARGET
from data import failure_stratum, load_train_test

df = pd.read_csv(CLEAN_DATA)
stratum = failure_stratum(df)
stratum.value_counts().rename("count").to_frame()

,count
none,9643
HDF,115
PWF,91
OSF,78
TWF,46


## 1. Split and save the row IDs

In [2]:
train_ids, test_ids = train_test_split(df["UDI"], test_size=TEST_SIZE, stratify=stratum, random_state=SEED)

train_ids.sort_values().to_frame().to_csv(TRAIN_IDS, index=False)
test_ids.sort_values().to_frame().to_csv(TEST_IDS, index=False)
print(f"Train: {len(train_ids)} jobs, test: {len(test_ids)} jobs")

Train: 6981 jobs, test: 2992 jobs


## 2. Verify the split

In [3]:
train, test = load_train_test()

checks = pd.Series({
    "No overlap between sets": set(train["UDI"]).isdisjoint(test["UDI"]),
    "All jobs assigned": len(train) + len(test) == len(df),
    "Loaded sets match saved IDs": set(train["UDI"]) == set(train_ids) and set(test["UDI"]) == set(test_ids),
})
checks

No overlap between sets        True
All jobs assigned              True
Loaded sets match saved IDs    True
dtype: bool

In [4]:
def composition(column):
    table = pd.DataFrame({
        "train count": column(train).value_counts(),
        "test count": column(test).value_counts(),
    }).fillna(0).astype(int)
    table["train share"] = (table["train count"] / len(train)).round(4)
    table["test share"] = (table["test count"] / len(test)).round(4)
    return table


composition(failure_stratum)

,train count,test count,train share,test share
none,6750,2893,0.9669,0.9669
HDF,80,35,0.0115,0.0117
PWF,64,27,0.0092,0.0090
OSF,55,23,0.0079,0.0077
TWF,32,14,0.0046,0.0047


In [5]:
composition(lambda d: d[TARGET])

,train count,test count,train share,test share
Machine failure,,,,
0,6750,2893,0.9669,0.9669
1,231,99,0.0331,0.0331


In [6]:
composition(lambda d: d["Type"])

,train count,test count,train share,test share
Type,,,,
L,4190,1794,0.6002,0.5996
M,2096,895,0.3002,0.2991
H,695,303,0.0996,0.1013


Both sets have the same failure rate, the same mix of failure modes and the same mix of product types, so results on the test set are representative of the whole dataset.